In [4]:
from grindstone import *
from src.grindstone import consts, pipeline
import grindstone.stage as stage
import grindstone.stage.image
import grindstone.stage.adjust
import grindstone.stage.load
import grindstone.stage.parse
import grindstone.stage.process
import grindstone.stage.render
#dirname = "/home/philo/Programming/linecam/basler/05-29_12-25"
# use direction NZ, units = 10
# dirname = "/home/philo/Programming/linecam/basler/05-29_14-41" # NZ, u = 1000
dirname = "/home/philo/Programming/linecam/basler/05-29_15-00"
velocity = 50.0
unitspersample = 1000.0
argdir = consts.AccelDirection.NZ
# accum_strategy = stage.image.ColorFramesFromTimestampsSquare(
#             invert=True, width=2098, dirname=dirname,
#         )
accum_strategy = stage.image.ColorFramesFromTimestampsBig(invert=True, width=10000, height=2098, dirname=dirname, ext="jpeg")
pipe = pipeline.Pipeline(stages=[
        stage.load.DirectoryLoad(dirname=dirname),
        stage.load.MetadataCSV(dirname=dirname),
        stage.parse.DirectoryStringData(
            argdir=argdir,
            calibration=[0.0, 0.0, 0.0],
            gravity=consts.Gravity,
        ),
        stage.process.GenerateSpeedsFromAccelerometer(),
        stage.adjust.InitialVelocity(velocity),
        stage.process.GeneratePositionsFromSpeed(),
        stage.process.GenerateTimestampsPerFrame(unitspersample),
        stage.load.MmapColorImage(dirname),
        stage.render.SelectColorFramesFromTimestamps(
            accum_strategy,
            1.0/unitspersample,
        ),
    ])

so the goal at this point is to run a pipeline through and display a segment. in order to do that, i need to figure out what the hell Maddie's code is trying to do here

In [5]:
pipe

Pipeline(stages=[DirectoryLoad(dirname='/home/philo/Programming/linecam/basler/05-29_15-00'), MetadataCSV(dirname='/home/philo/Programming/linecam/basler/05-29_15-00'), DirectoryStringData(argdir=<AccelDirection.NZ: 8>, calibration=[0.0, 0.0, 0.0], gravity=9.80665), GenerateSpeedsFromAccelerometer(), InitialVelocity(initialVelocity=50.0), GeneratePositionsFromSpeed(), GenerateTimestampsPerFrame(metersPerPixel=1000.0), MmapColorImage(dirname='/home/philo/Programming/linecam/basler/05-29_15-00'), SelectColorFramesFromTimestamps(accum_strategy=<grindstone.stage.image.ColorFramesFromTimestampsBig object at 0x7ff478ab2710>, distance=0.001)])

ceci _est_ un pipe

In [6]:
pipe.run()

entering DirectoryLoad(dirname='/home/philo/Programming/linecam/basler/05-29_15-00')
entering MetadataCSV(dirname='/home/philo/Programming/linecam/basler/05-29_15-00')
entering DirectoryStringData(argdir=<AccelDirection.NZ: 8>, calibration=[0.0, 0.0, 0.0], gravity=9.80665)
time unit is microseconds, converting to milliseconds
entering GenerateSpeedsFromAccelerometer()
accel=[0.102783  0.10083   0.0993652 ... 0.0151367 0.0164795 0.0214844]
entering InitialVelocity(initialVelocity=50.0)
entering GeneratePositionsFromSpeed()
entering GenerateTimestampsPerFrame(metersPerPixel=1000.0)
0 days 00:00:50.102783               0.000
0 days 00:01:40.306396               0.924
0 days 00:02:30.609374200            1.941
0 days 00:03:21.014159400            3.005
0 days 00:04:11.515380100            3.991
                                   ...    
3572 days 07:37:40.212886870    152576.998
3572 days 09:04:30.029020429    152578.060
3572 days 10:31:19.860290706    152579.114
3572 days 11:58:09.7080404

for `05-29_14-41`:  
`p < lP` for 26459 positions  
`p > lP` for 99934 positions  
there are 128193 acceleration samples, and _~20%_ of them are moving in the wrong direction, but only in this one capture

if it's only happening on this one capture, I'm tempted not to fix it